# Numpy 之六：索引与切片

---

**索引 = 取一个元素；切片 = 取一段元素。**
这是数据处理里最高频的操作，没有之一。

## 学习目标

1. 掌握 3 种取数方式：**整数索引 / 切片 / 布尔索引**
2. 掌握二维数组的"行列定位"
3. 理解切片返回的是**视图**（改切片会影响原数组！）
4. 认识**花式索引**（用下标数组取数）


In [ ]:
import numpy as np

arr = np.array([1, 2, 3, 4, 5])
print('arr =', arr)


---
## 一、整数索引（取单个元素）

**下标从 0 开始**，这和 Python list 一样。


In [ ]:
print('arr[0] =', arr[0])
print('arr[3] =', arr[3])
print('arr[-1] =', arr[-1], '  <- 负数表示"从后往前数"，-1 是最后一个')
print('arr[-2] =', arr[-2])


---
## 二、切片：`[start : stop : step]`

**核心规则：`start` 取得到，`stop` 取不到（左闭右开）。**

```
arr[1:4]  ->  下标 1, 2, 3  （不包含 4）
```

记法：**"含头不含尾"**。


In [ ]:
print('arr[1:4]   =', arr[1:4], '   <- 下标 1,2,3')
print('arr[:3]    =', arr[:3], '   <- 省略 start，从头开始')
print('arr[2:]    =', arr[2:], '   <- 省略 stop，一直到末尾')
print('arr[:]     =', arr[:], '   <- 从头到尾（整份）')
print('arr[::2]   =', arr[::2], '   <- 步长为 2，隔一个取一个')
print('arr[1::2]  =', arr[1::2], '   <- 从 1 开始，隔一个取一个')
print('arr[::-1]  =', arr[::-1], '   <- 步长为 -1，倒序！')


> **正着数：通过 `[::-1]` 倒序** 是 Python 里翻转序列的标准写法，比 `reversed()` 简洁。


### ⚠️ 重要：切片是"视图"，不是"副本"

这句话的意思是：**切片和原数组共用同一块内存**。
所以**改切片，原数组也会变**。

这是 Numpy 为了"快"做的设计（避免没必要的复制），但也是新手最容易踩的坑。


In [ ]:
a = np.array([1, 2, 3, 4, 5])
s = a[1:4]
print('a      =', a)
print('s = a[1:4] =', s)

s[0] = 999                # 改切片
print('\n改 s[0] = 999 之后：')
print('  s =', s)
print('  a =', a, '  <- 原数组也变了！')


**想要真正独立的副本？** 用 `.copy()`：


In [ ]:
a2 = np.array([1, 2, 3, 4, 5])
s2 = a2[1:4].copy()        # 显式复制

s2[0] = 999
print('用了 .copy() 之后：')
print('  s2 =', s2)
print('  a2 =', a2, '  <- 原数组没变 ✅')


> **怎么判断是视图还是副本？** 看它的 `.base`：
> - `.base is 原数组` -> 视图（共享内存）
> - `.base is None` -> 副本（独立内存）


In [ ]:
print('切片 .base 是 a 吗？', np.array([1, 2, 3, 4, 5])[1:4].base is not None, '（视图）')
print('copy  .base 是 None 吗？', np.array([1, 2, 3, 4, 5])[1:4].copy().base is None, '（副本）')


---
## 三、二维数组的索引：用逗号分隔"行"和"列"

写法：`arr[行, 列]`。

**这是 Numpy 比 list 优雅的地方** —— list 要写 `arr[0][1]`，Numpy 直接 `arr[0, 1]`。


In [ ]:
m = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])
print('矩阵 m：')
print(m)

print('\nm[0, 1] =', m[0, 1], '  <- 第 0 行第 1 列（值是 2）')
print('m[2, 2] =', m[2, 2], '  <- 第 2 行第 2 列（值是 9）')


### 取一行 / 取一列

```
m[0, :]     -> 第 0 行（所有列）  -> 一维数组
m[:, 1]     -> 第 1 列（所有行）  -> 一维数组
```


In [ ]:
print('m[0, :] 取第 0 行 :', m[0, :])
print('m[:, 1] 取第 1 列 :', m[:, 1])

# 冒号也可以省略（只写行）
print('m[0]    等价于 m[0, :] :', m[0])


> **注意形状**：`m[0]` 得到的是**一维**数组 `(3,)`，
> 不是 `(1, 3)` 的二维。想要二维的"一行"，用切片 `m[0:1]`：


In [ ]:
print('m[0].shape    =', m[0].shape, '   <- 一维')
print('m[0:1].shape  =', m[0:1].shape, '   <- 二维，保留了"行"这个维度')


### 行列都可以切片


In [ ]:
print('m[0:2, 1:3] =')
print(m[0:2, 1:3])
print('  <- 第 0~1 行、第 1~2 列 的子矩阵')

print('\nm[1:3, 2] =', m[1:3, 2], '  <- 第 1~2 行、第 2 列那一竖条')
print('m[:, 1:3] =')
print(m[:, 1:3])
print('  <- 所有行、第 1~2 列')


---
## 四、布尔索引（最实用的筛选方式）

把"条件判断"产生的布尔数组当作下标，就能取出所有满足条件的元素。

**这是数据分析的日常动作**：`df[df["age"] > 18]` 就是这个原理。


In [ ]:
scores = np.array([55, 78, 92, 43, 88, 67])
print('成绩：', scores)

mask = scores >= 60
print('布尔掩码 mask =', mask)
print('及格分数：', scores[mask])
print('不及格分数：', scores[~mask], '  <- ~ 是"取反"')

# 直接写条件也行
print('\n大于 80 的：', scores[scores > 80])


In [ ]:
# 结合多个条件：一定要加括号！
print('60 到 90 之间的：', scores[(scores >= 60) & (scores <= 90)])

# 布尔索引还能用来【赋值】
s2 = scores.copy()
s2[s2 < 60] = 0            # 把不及格的都改成 0
print('\n把不及格的改成 0：', s2)


### 二维数组的布尔筛选（返回一维结果）


In [ ]:
print('m 里大于 5 的元素：', m[m > 5])
print('  <- 注意返回的是【一维】数组，因为"满足条件的元素"本身没有固定形状')

# 按行筛选：第 1 列大于 3 的那些行
print('\n第 1 列大于 3 的行：')
print(m[m[:, 1] > 3])


---
## 五、花式索引（fancy indexing）

用**下标数组**去取值，可以一次取多个任意位置的元素，
也可以用来"重排顺序"。


In [ ]:
a = np.array([10, 20, 30, 40, 50])

# 取出下标 0、2、4 的元素
print('a[[0, 2, 4]] =', a[[0, 2, 4]])

# 取出的顺序由下标数组决定 -> 可以用来重排
idx = [4, 3, 2, 1, 0]
print('a[[4,3,2,1,0]] =', a[idx], '  <- 相当于倒序')

# 花式索引【一定返回副本】，不是视图
fancy = a[[0, 1]]
fancy[0] = 999
print('\n花式索引取的可以随便改，不影响原数组：', a, 'vs', fancy)


In [ ]:
# 二维的花式索引：取指定行
print('取第 0 行和第 2 行：')
print(m[[0, 2]])

# 取指定行列组合：(0,0), (1,1), (2,2) 对角线
print('\n取对角线 (0,0),(1,1),(2,2)：', m[[0, 1, 2], [0, 1, 2]])


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| `arr[1:4]` 以为是 1,2,3,4 | 少了最后一个 | **含头不含尾**，`stop` 取不到 |
| 改切片后发现原数组变了 | 数据被"共享" | 切片是**视图**，要独立用 `.copy()` |
| `m[0]` 以为是二维的一行 | 形状是 `(3,)` | 要二维用 `m[0:1]` |
| 多条件不加括号 | 报错或结果诡异 | `(a > 1) & (a < 5)` |
| 用 `and` / `or` 连条件 | `ValueError` | 数组要用 `&` `|` |
| 花式索引当视图 | 结果对但机制理解错 | 花式索引**返回副本** |
| 布尔索引想保持原形状 | 返回一维 | 布尔筛选天然"拍平"，需要形状用 `np.where` |
| 负索引越界 | `IndexError` | `arr[-1]` 是最后一个，`arr[-len:]` 才是整个 |

---

## 七、动手练习

**练习 1**：`a = np.arange(10)`，取出：前 5 个、后 3 个、所有偶数位、倒序。

**练习 2**：`m = np.arange(1, 13).reshape(3, 4)`，
取出第 2 行、第 1 列、右下角 2×2 子矩阵。

**练习 3**：生成 10 个随机整数，挑出其中所有大于 50 的，并把它们替换成 0。

**练习 4**：`a = np.array([5, 3, 9, 1])`，用花式索引把它变成 `[9, 5, 3, 1]`（降序）。

**练习 5**：`m` 是 `(5, 3)` 的矩阵，取出"第 3 列大于 10"的所有行。

**练习 6**：验证 `a[1:3]` 是视图，而 `a[[1, 2]]` 是副本。

---

**答案思路（先自己做）**

```python
# 1
a = np.arange(10)
print(a[:5], a[-3:], a[::2], a[::-1])

# 2
m = np.arange(1, 13).reshape(3, 4)
print(m[1], m[:, 1], m[1:3, 2:4])

# 3
x = np.random.randint(0, 100, 10)
x[x > 50] = 0

# 5
m = np.random.randint(0, 20, (5, 3))
print(m[m[:, 2] > 10])
```
